# Setup

In [15]:
!pip install opik pandas


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [16]:
import pandas as pd
import json
import os
import re
from opik import Opik
from opik.evaluation.metrics import base_metric, score_result
from opik.integrations.openai import track_openai
from opik.evaluation import evaluate
from openai import OpenAI
from typing import Any

# Opik Configuration
os.environ["OPIK_URL_OVERRIDE"] = "https://3.110.54.210/api"
os.environ["OPIK_CHECK_TLS_CERTIFICATE"] = "false"
if not os.environ.get("OPENROUTER_API_KEY"):
    raise ValueError("Set OPENROUTER_API_KEY in your environment before running this notebook")
os.environ["OPIK_PROJECT_NAME"] = "AI Evaluations"

# Experiment Configuration
EXPERIMENT_NAME = "gempro-eval_google/openai/gpt-5.4-mini"
MODEL_NAME = "openai/gpt-5.4-mini"
DATASET_NAME = "ai_tutor_coding_practise"

# OpenRouter client
openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)

# Opik client
opik_client = Opik(project_name=os.environ["OPIK_PROJECT_NAME"])

print("✅ Environment configured")

✅ Environment configured


# Upload Dataset

In [18]:
dataset = opik_client.get_or_create_dataset(DATASET_NAME)
print(dataset)

# Fetch all items (if small) and count
items = dataset.get_items()
current_size = len(items)

if current_size == 0:
    df = pd.read_csv("your_dataset.csv")
    dataset.insert_from_pandas(dataframe=df)
    print(f"✅ Uploaded {len(df)} rows to '{DATASET_NAME}'")
else:
    print(f"✅ Dataset '{DATASET_NAME}' already exists with {current_size} rows, skipping upload")

✅ Dataset 'ai_tutor_coding_practise' already exists with 100 rows, skipping upload


# Set Up Model Config

In [19]:
# ── All metrics use this shared evaluator LLM ──────────────────────────────
EVALUATOR_MODEL = "google/gemini-2.5-pro"
def call_evaluator(prompt: str) -> dict:
    response = openrouter_client.chat.completions.create(
        model=EVALUATOR_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.001,
    )
    raw = response.choices[0].message.content.strip()
    raw = re.sub(r"```json|```", "", raw).strip()
    return json.loads(raw)

print(f"✅ Evaluator LLM configured: {EVALUATOR_MODEL}")

✅ Evaluator LLM configured: google/gemini-2.5-pro


# Create you custom evaluators

# ClosingStatementDetectionMetric

In [20]:
class ClosingStatementDetectionMetric(base_metric.BaseMetric):
    """
    Scores -1 if student input is NOT a closing statement.
    Scores  1 if it IS a closing statement AND response handles it correctly.
    Scores  0 if it IS a closing statement BUT response fails criteria.
    """
    def __init__(self, name: str = "closing_statement_detection"):
        self.name = name
        self.prompt_template = """
You are evaluating whether the AI tutor correctly detected and responded to a student's closing statement in a video session tutoring context.

A closing statement is when the student expresses satisfaction and wants to conclude the current conversation (e.g., "thank you", "got it", "understood", "thanks", "okay got it").

## Evaluation Criteria:
When the student input IS a closing statement, the response MUST meet ALL of the following:
- Correctly identifies the satisfaction/conclusion signal
- Provides a brief, encouraging closing response (10-15 words max)
- Sets should_student_reply: "No"
- Sets reason_flag: "No"
- Does NOT continue explaining or ask follow-up questions
- Uses varied responses

## Context Provided:
Previous Conversation: {prev_conversation}
Student Input: {student_input}
Tutor Response to Evaluate: {output}

## Scoring:
Score -1 if the student input is NOT a closing statement.
Score  1 if the student input IS a closing statement AND the response correctly handles it by meeting ALL criteria listed above.
Score  0 if the student input IS a closing statement BUT the response fails ANY criteria.

Be extremely strict. Even one violation means the score is 0.

Return your evaluation as a JSON object:
{{
    "reason": "<detailed explanation with specific examples from the response>",
    "score": <-1 or 0 or 1>
}}
Return only valid JSON with exactly two keys: reason and score.
"""

    def score(self, output: str, prev_conversation: str = "", input_messages: list = None, **ignored_kwargs: Any):
        student_input = ""
        if input_messages:
            for msg in reversed(input_messages):
                if isinstance(msg, dict) and msg.get("role") == "user":
                    student_input = msg.get("content", "")
                    break

        prompt = self.prompt_template.format(
            prev_conversation=prev_conversation,
            student_input=student_input,
            output=output,
        )
        result = call_evaluator(prompt)
        return score_result.ScoreResult(
            name=self.name,
            value=float(result["score"]),
            reason=result["reason"],
        )

print("✅ ClosingStatementDetectionMetric defined")

✅ ClosingStatementDetectionMetric defined


# ResponseFormatComplianceMetric

In [21]:
class ResponseFormatComplianceMetric(base_metric.BaseMetric):
    """
    Scores 1 if response strictly follows all JSON formatting requirements.
    Scores 0 if response violates ANY requirement.
    """
    def __init__(self, name: str = "response_format_compliance"):
        self.name = name
        self.prompt_template = """
You are evaluating whether the AI tutor response follows the required JSON format structure and content guidelines for video session tutoring.

## Required Format:
{{
  "reply": "<text only, no code, 40-50 words max>",
  "should_student_reply": "Yes" / "No",
  "reason_flag": "Yes" / "No"
}}

## Evaluation Criteria:
The response MUST meet ALL of the following:
- Valid JSON structure (starts with {{, ends with }})
- Contains exactly three keys: reply, should_student_reply, reason_flag
- No markdown formatting (no triple backticks, no ```json)
- No code blocks or code fences around the JSON
- No extra text or characters outside the JSON object
- The reply field contains NO code snippets or syntax examples
- should_student_reply value is either "Yes" or "No" (not true/false or 1/0)
- reason_flag value is either "Yes" or "No" (not true/false or 1/0)

## Context Provided:
Previous Conversation: {prev_conversation}
Student Input: {student_input}
Tutor Response to Evaluate: {output}

## Scoring:
Score 1 if the response strictly follows ALL formatting requirements.
Score 0 if the response violates ANY requirement.

Be extremely strict. Even a single violation means score is 0.

Return your evaluation as a JSON object:
{{
    "reason": "<detailed explanation with specific examples from the response>",
    "score": <0 or 1>
}}
Return only valid JSON with exactly two keys: reason and score.
"""

    def score(self, output: str, prev_conversation: str = "", input_messages: list = None, **ignored_kwargs: Any):
        student_input = ""
        if input_messages:
            for msg in reversed(input_messages):
                if isinstance(msg, dict) and msg.get("role") == "user":
                    student_input = msg.get("content", "")
                    break

        prompt = self.prompt_template.format(
            prev_conversation=prev_conversation,
            student_input=student_input,
            output=output,
        )
        result = call_evaluator(prompt)
        return score_result.ScoreResult(
            name=self.name,
            value=float(result["score"]),
            reason=result["reason"],
        )

print("✅ ResponseFormatComplianceMetric defined")

✅ ResponseFormatComplianceMetric defined


# ResponseQualityMetric

In [22]:
class ResponseQualityMetric(base_metric.BaseMetric):
    """
    Scores 1 if response meets ALL communication and explanation quality criteria.
    Scores 0 if response fails ANY criterion.
    """
    def __init__(self, name: str = "response_quality"):
        self.name = name
        self.prompt_template = """
You are evaluating the quality of the AI tutor's response in a video session tutoring context, focusing on tone, clarity, and communication effectiveness.

## Evaluation Criteria:
The response MUST meet ALL of the following:

Communication Quality:
- Response length: 40-50 words maximum
- Always responds in English (regardless of student's query language)
- Supportive, encouraging tone (never harsh or dismissive)
- Natural, conversational language (filler words like "Hmm", "I see", "Okay" are acceptable)
- Sounds human-like, not robotic

Explanation Quality:
- Clear conceptual explanations without code snippets
- Uses analogies or real-world examples to explain concepts
- Acknowledges student's understanding level and progress
- No syntax, code examples, or programming instructions
- Focuses on helping student understand "why", not just "what"

## Context Provided:
Previous Conversation: {prev_conversation}
Student Input: {student_input}
Tutor Response to Evaluate: {output}

## Scoring:
Score 1 if the response meets ALL criteria listed above.
Score 0 if the response fails ANY criterion.

Be extremely strict. Even one violation means score is 0.

Return your evaluation as a JSON object:
{{
    "reason": "<detailed explanation with specific examples from the response>",
    "score": <0 or 1>
}}
Return only valid JSON with exactly two keys: reason and score.
"""

    def score(self, output: str, prev_conversation: str = "", input_messages: list = None, **ignored_kwargs: Any):
        student_input = ""
        if input_messages:
            for msg in reversed(input_messages):
                if isinstance(msg, dict) and msg.get("role") == "user":
                    student_input = msg.get("content", "")
                    break

        prompt = self.prompt_template.format(
            prev_conversation=prev_conversation,
            student_input=student_input,
            output=output,
        )
        result = call_evaluator(prompt)
        return score_result.ScoreResult(
            name=self.name,
            value=float(result["score"]),
            reason=result["reason"],
        )

print("✅ ResponseQualityMetric defined")

✅ ResponseQualityMetric defined


# IntentUnderstandingMetric

In [23]:
class IntentUnderstandingMetric(base_metric.BaseMetric):
    """
    Scores 1 if the tutor correctly understood the student's intent.
    Scores 0 if the tutor misunderstood or answered a different question.
    """
    def __init__(self, name: str = "intent_understanding"):
        self.name = name
        self.prompt_template = """
You are evaluating whether the AI tutor correctly understood the student's intended question, even when the question is vague, unclear, or grammatically incorrect.

## Evaluation Criteria:
The response MUST demonstrate ANY of the following:
- Addresses what the student actually meant, not just what they literally said
- Interprets vague or broken sentences appropriately based on context
- Uses previous conversation history to clarify the current query
- Handles grammar mistakes, transcription errors, or incomplete thoughts
- Asks appropriate clarifying questions when intent is genuinely unclear
- Makes reasonable assumptions based on video content and context

## Context Provided:
Previous Conversation: {prev_conversation}
Student Input: {student_input}
Tutor Response to Evaluate: {output}

## Scoring:
Score 1 if the response demonstrates correct understanding of student's intent (meets ANY of the criteria above).
Score 0 if the response misunderstands the intent, answers a different question, or fails to address the core confusion.

Be extremely strict. If the response clearly misunderstands what the student was asking, score is 0.

Return your evaluation as a JSON object:
{{
    "reason": "<detailed explanation with specific examples from the response>",
    "score": <0 or 1>
}}
Return only valid JSON with exactly two keys: reason and score.
"""

    def score(self, output: str, prev_conversation: str = "", input_messages: list = None, **ignored_kwargs: Any):
        student_input = ""
        if input_messages:
            for msg in reversed(input_messages):
                if isinstance(msg, dict) and msg.get("role") == "user":
                    student_input = msg.get("content", "")
                    break

        prompt = self.prompt_template.format(
            prev_conversation=prev_conversation,
            student_input=student_input,
            output=output,
        )
        result = call_evaluator(prompt)
        return score_result.ScoreResult(
            name=self.name,
            value=float(result["score"]),
            reason=result["reason"],
        )

print("✅ IntentUnderstandingMetric defined")

✅ IntentUnderstandingMetric defined


# BoundaryEnforcementMetric

In [24]:
class BoundaryEnforcementMetric(base_metric.BaseMetric):
    """
    Scores -1 if student query is on-topic (metric not applicable).
    Scores  1 if query is off-topic AND response correctly handles it.
    Scores  0 if query is off-topic BUT response fails to handle it correctly.
    """
    def __init__(self, name: str = "boundary_enforcement"):
        self.name = name
        self.prompt_template = """
You are evaluating whether the AI tutor correctly identifies and handles off-topic, inappropriate, or out-of-scope queries in a video session tutoring context.

## Evaluation Criteria:

Off-Topic Detection — response MUST meet ALL relevant criteria when query is off-topic:
- Correctly identifies queries unrelated to Python learning or video session content
- Identifies nonsensical/random inputs (e.g., "blue", "hello", gibberish)
- Politely declines and redirects student back to video content
- Sets reason_flag: "Yes" for off-topic queries
- Does NOT engage with or answer off-topic requests

Assessment/Homework Protection:
- Refuses to provide direct answers to assessment questions
- Refuses to provide direct solutions to homework problems
- Offers only conceptual guidance, not complete solutions
- Redirects to understanding concepts rather than getting answers

## Context Provided:
Previous Conversation: {prev_conversation}
Student Input: {student_input}
Tutor Response to Evaluate: {output}

## Scoring:
Score -1 if the student query is on-topic and appropriate (related to Python learning or video content).
Score  1 if the query is off-topic/inappropriate AND the response correctly handles it by meeting ALL relevant criteria above.
Score  0 if the query is off-topic/inappropriate BUT the response fails to handle it correctly.

Be extremely strict.

Return your evaluation as a JSON object:
{{
    "reason": "<detailed explanation with specific examples from the response>",
    "score": <-1 or 0 or 1>
}}
Return only valid JSON with exactly two keys: reason and score.
"""

    def score(self, output: str, prev_conversation: str = "", input_messages: list = None, **ignored_kwargs: Any):
        student_input = ""
        if input_messages:
            for msg in reversed(input_messages):
                if isinstance(msg, dict) and msg.get("role") == "user":
                    student_input = msg.get("content", "")
                    break

        prompt = self.prompt_template.format(
            prev_conversation=prev_conversation,
            student_input=student_input,
            output=output,
        )
        result = call_evaluator(prompt)
        return score_result.ScoreResult(
            name=self.name,
            value=float(result["score"]),
            reason=result["reason"],
        )

print("✅ BoundaryEnforcementMetric defined")

✅ BoundaryEnforcementMetric defined


# Assemble Metrics

In [25]:
coding_practise_metrics = [
    ClosingStatementDetectionMetric(),
    ResponseFormatComplianceMetric(),
    ResponseQualityMetric(),
    IntentUnderstandingMetric(),
    BoundaryEnforcementMetric(),
]

print(f"✅ {len(coding_practise_metrics)} metrics ready:")
for m in coding_practise_metrics:
    print(f"   • {m.name}")

✅ 5 metrics ready:
   • closing_statement_detection
   • response_format_compliance
   • response_quality
   • intent_understanding
   • boundary_enforcement


# Experiment — Run Evaluation

In [ ]:
dataset = opik_client.get_dataset(name=DATASET_NAME)
tracked_llm_client = track_openai(openrouter_client)

def llm_call(dataset_item: dict) -> str:
    try:
        conversation = json.loads(dataset_item["input"])
        prompt_template = dataset_item["current_prompt"]
        messages = [{"role": "system", "content": prompt_template}]
        for msg in conversation:
            if isinstance(msg, dict) and msg.get("role") in ["user", "assistant"]:
                messages.append({"role": msg["role"], "content": msg["content"]})
        response = tracked_llm_client.chat.completions.create(
            model=MODEL_NAME,
            messages=messages,
            temperature=0.001,
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error: {str(e)}"

def evaluation_task(dataset_item: dict) -> dict:
    output = llm_call(dataset_item)
    try:
        conversation = json.loads(dataset_item["input"])
    except Exception:
        conversation = []

    prev_conversation = ""
    if conversation:
        last_user_idx = -1
        for i in range(len(conversation) - 1, -1, -1):
            if isinstance(conversation[i], dict) and conversation[i].get("role") == "user":
                last_user_idx = i
                break
        span = conversation[:last_user_idx] if last_user_idx > 0 else conversation[:-1]
        prev_lines = [
            f"{msg['role']}: {msg.get('content', '')}"
            for msg in span
            if isinstance(msg, dict) and msg.get("role") in ["user", "assistant"]
        ]
        prev_conversation = "\n".join(prev_lines)

    return {
        "output": output,
        "prev_conversation": prev_conversation,
        "extracted_question": dataset_item.get("question_content", "") or "",
        "input_messages": conversation,
    }

print(f"🚀 Starting evaluation : {EXPERIMENT_NAME}")
print(f"📁 Dataset             : {DATASET_NAME}")
print(f"🤖 Model               : {MODEL_NAME}")
print(f"📊 Metrics             : {len(coding_practise_metrics)}")
print(f"Evaluate object : {evaluate}")

dataset.get_version_info = lambda: None

eval_results = evaluate(
    experiment_name=EXPERIMENT_NAME,
    dataset=dataset,
    task=evaluation_task,
    scoring_metrics=coding_practise_metrics,
    task_threads=2,
    verbose=1,
)

print("✅ Evaluation completed")

🚀 Starting evaluation : gempro-eval_google/openai/gpt-5.4-mini
📁 Dataset             : ai_tutor_coding_practise
🤖 Model               : openai/gpt-5.4-mini
📊 Metrics             : 5
Evaluate object : <function evaluate at 0x70aa7b3d60c0>


Evaluation (there might be a delay before the first items are processed): 0it [00:00, ?it/s]

OPIK: Failed to compute metric response_quality. Score result will be marked as failed.
Traceback (most recent call last):
  File "/home/nxtwave/Desktop/CodeBase-HUB/Opik/.venv/lib/python3.12/site-packages/opik/evaluation/engine/metrics_evaluator.py", line 180, in _compute_metric_scores
    result = metric.score(**mapped_scoring_inputs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_39396/2051926440.py", line 60, in score
    result = call_evaluator(prompt)
             ^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_39396/3910948502.py", line 9, in call_evaluator
    raw = response.choices[0].message.content.strip()
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'NoneType' object has no attribute 'strip'
